# Complexity-04b — Secrétaire et k-server en ligne

[Source Python](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) · [Index](README.md) · [04d](Complexity-04d-Secretaire-Matroidal-Python.ipynb)

**Objectifs** : mesurer la règle du secrétaire, comparer deux oracles de matroïde et deux stratégies, puis calculer exactement WFA et OPT sur ligne et cycle. **Prérequis** : ensembles, permutations et coût en ligne. **Durée pédagogique** : 45 minutes.

Les simulations sont une transposition C# native du carnet Python versionné, qui est la source primaire consultée ici. Les annonces de Singla et de Coester et collaborateurs sont rapportées par ce carnet, sans lecture directe des prépublications. Les sorties dépendent du générateur .NET : les graines rendent les essais reproductibles dans cet environnement, mais ne reproduisent pas la suite NumPy.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
Console.WriteLine("Structures et générateur .NET prêts");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Structures et générateur .NET prêts


## 1. Secrétaire classique

On observe les ⌊n/e⌋ premières personnes et retient ensuite le premier record. La limite 1/e est asymptotique ; le banc garde les trois tailles et 20 000 essais par taille.

In [2]:
var rng = new Random(42);
int[] Permutation(int n, Random r) {
    var p = Enumerable.Range(0, n).ToArray();
    for (int i = n - 1; i > 0; i--) { int j = r.Next(i + 1); (p[i], p[j]) = (p[j], p[i]); }
    return p;
}
bool Secretaire(int[] p) {
    int seuil = Math.Max(1, (int)(p.Length / Math.E));
    int meilleur = p.Take(seuil).Max();
    foreach (int x in p.Skip(seuil)) if (x > meilleur) return x == p.Length - 1;
    return false;
}
double MonteCarlo(int n, int essais, Random r) {
    int succes = 0;
    for (int i = 0; i < essais; i++) if (Secretaire(Permutation(n, r))) succes++;
    return (double)succes / essais;
}
foreach (int n in new[]{20,100,1000})
    Console.WriteLine($"n={n} : {MonteCarlo(n, 20000, rng):F4} (limite 1/e={1/Math.E:F4})");

n=20 : 0.3833 (limite 1/e=0.3679)


n=100 : 0.3771 (limite 1/e=0.3679)


n=1000 : 0.3690 (limite 1/e=0.3679)


La comparaison porte sur des probabilités empiriques ; un tirage fini ne prouve pas l'optimalité asymptotique.

## 2. Matroïdes et stratégies

L'oracle uniforme borne le cardinal ; l'oracle graphique détecte un cycle. Le glouton hors ligne fournit l'optimum pour ces matroïdes. Les deux stratégies en ligne sont des références, pas l'algorithme de Singla.

In [3]:
bool IndepUniforme(IEnumerable<int> indices, int rang) => indices.Count() <= rang;
bool IndepGraphique(IEnumerable<int> indices, (int,int)[] aretes) {
    var parent = new Dictionary<int,int>();
    int Racine(int x) { if (!parent.ContainsKey(x)) parent[x] = x; while (parent[x] != x) x = parent[x]; return x; }
    foreach (int i in indices) {
        var (a,b) = aretes[i]; int ra = Racine(a), rb = Racine(b);
        if (ra == rb) return false;
        parent[ra] = rb;
    }
    return true;
}
(double,List<int>) Optimum(double[] valeurs, Func<IEnumerable<int>,bool> ind) {
    var choix = new List<int>();
    foreach (int i in Enumerable.Range(0,valeurs.Length).OrderByDescending(i => valeurs[i])) {
        var candidat = choix.Append(i);
        if (ind(candidat)) choix.Add(i);
    }
    return (choix.Sum(i => valeurs[i]), choix);
}
double Strategie(double[] valeurs, Func<IEnumerable<int>,bool> ind, bool seuil, Random r) {
    var ordre = Permutation(valeurs.Length, r); int debut = seuil ? valeurs.Length / 2 : 0;
    double mediane = 0;
    if (seuil) { var echantillon = ordre.Take(debut).Select(i=>valeurs[i]).OrderBy(x=>x).ToArray();
        mediane = (echantillon[(debut-1)/2] + echantillon[debut/2]) / 2; }
    var choix = new List<int>();
    foreach (int i in ordre.Skip(debut)) if ((!seuil || valeurs[i] >= mediane) && ind(choix.Append(i))) choix.Add(i);
    return choix.Sum(i=>valeurs[i]);
}
double Competitivite(double[] valeurs, Func<IEnumerable<int>,bool> ind, bool seuil, int essais, Random r) {
    double total = 0, opt = Optimum(valeurs,ind).Item1;
    for (int i=0;i<essais;i++) total += Strategie(valeurs,ind,seuil,r);
    return total / essais / opt;
}
var valU = Enumerable.Range(0,12).Select(_=>1+9*rng.NextDouble()).ToArray();
var ar = new List<(int,int)>();
for (int l=0;l<3;l++) for (int c=0;c<4;c++) { int v=l*4+c; if(c<3) ar.Add((v,v+1)); if(l<2) ar.Add((v,v+4)); }
var aretes = ar.ToArray(); var valG = aretes.Select(_=>1+9*rng.NextDouble()).ToArray();
Func<IEnumerable<int>,bool> uniforme = s => IndepUniforme(s,3);
Func<IEnumerable<int>,bool> graphique = s => IndepGraphique(s,aretes);
foreach (var inst in new[]{("uniforme",valU,uniforme),("graphique",valG,graphique)}) {
    Console.WriteLine($"{inst.Item1} OPT={Optimum(inst.Item2,inst.Item3).Item1:F2} ; pur={Competitivite(inst.Item2,inst.Item3,false,2000,rng):F4} ; seuil={Competitivite(inst.Item2,inst.Item3,true,2000,rng):F4}");
}
Console.WriteLine($"Grille 3×4 : {aretes.Length} arêtes ; 2000 essais par stratégie");

uniforme OPT=25.95 ; pur=0.6948 ; seuil=0.7265


graphique OPT=79.05 ; pur=0.7664 ; seuil=0.4391


Grille 3×4 : 17 arêtes ; 2000 essais par stratégie


Les deux oracles imposent des contraintes différentes. L'ordre et les poids tirés peuvent changer les ratios ; aucune comparaison ponctuelle ne garantit une dominance générale.

In [4]:
var graines = new[]{0,1,7,42,99,123,256,1024};
var purs = new List<double>(); var seuils = new List<double>();
foreach(int seed in graines) { var r = new Random(seed); purs.Add(Competitivite(valU,uniforme,false,2000,r)); seuils.Add(Competitivite(valU,uniforme,true,2000,r)); }
double EcartType(List<double> x) { double m=x.Average(); return Math.Sqrt(x.Average(v=>(v-m)*(v-m))); }
Console.WriteLine($"8 graines : pur [{purs.Min():F4}, {purs.Max():F4}], moyenne {purs.Average():F4}, écart-type {EcartType(purs):F4}");
Console.WriteLine($"8 graines : seuil [{seuils.Min():F4}, {seuils.Max():F4}], moyenne {seuils.Average():F4}, écart-type {EcartType(seuils):F4}; seuil devant {Enumerable.Range(0,8).Count(i=>seuils[i]>purs[i])}/8");

8 graines : pur [0.6946, 0.7061], moyenne 0.7001, écart-type 0.0034


8 graines : seuil [0.7123, 0.7427], moyenne 0.7274, écart-type 0.0080; seuil devant 8/8


Cette dispersion réutilise les mêmes poids et ne constitue pas une étude sur toutes les instances.

## Exercice 1 — Oracle de partition

Définir l'indépendance par quotas de couleur, puis évaluer la stratégie à seuil. Tester notamment l'ensemble vide, deux éléments de même couleur et deux couleurs distinctes.

In [5]:
bool? IndepPartition(int[] indices, int[] couleurs, int[] quotas) {
    // TODO : compter les éléments par couleur et comparer aux quotas.
    return null;
}
var ex1a=IndepPartition(new[]{0,1},new[]{0,0,1,1},new[]{1,1});
if(ex1a==null) Console.WriteLine("Exercice 1 à compléter");
else {
    if(ex1a!=false || IndepPartition(new[]{0,2},new[]{0,0,1,1},new[]{1,1})!=true || IndepPartition(Array.Empty<int>(),new[]{0,0,1,1},new[]{1,1})!=true)
        throw new Exception("Oracle de partition incorrect");
    Console.WriteLine("Oracle de partition : cas vide, conflit et deux couleurs vérifiés");
}

Exercice 1 à compléter


## 3. k-server : fonction de travail exacte

Pour deux serveurs distincts, une configuration est une paire triée. La DP calcule le coût minimal pour servir le préfixe et finir dans chaque configuration. WFA choisit sa configuration par coût immédiat plus fonction de travail, sans regarder les futures requêtes ; la DP finale donne OPT.

In [6]:
List<(int,int)> Configs(int m) {
    var c = new List<(int,int)>();
    for(int a=0;a<m;a++) for(int b=a+1;b<m;b++) c.Add((a,b));
    return c;
}
double DistanceConfig((int,int) a, (int,int) b, Func<int,int,int> d) =>
    Math.Min(d(a.Item1,b.Item1)+d(a.Item2,b.Item2), d(a.Item1,b.Item2)+d(a.Item2,b.Item1));
(double wfa,double opt) JouerWfa(int m,(int,int) init,int[] req,Func<int,int,int> d) {
    var configs=Configs(m); var w=configs.ToDictionary(c=>c,c=>DistanceConfig(init,c,d));
    var courant=init; double cout=0;
    foreach(int r in req) {
        var suivant=new Dictionary<(int,int),double>();
        foreach(var a in configs.Where(c=>c.Item1==r || c.Item2==r))
            suivant[a]=w.Min(kv=>kv.Value+DistanceConfig(kv.Key,a,d));
        // Le choix courant ne dépend que de la table du préfixe.
        var candidates=new List<(int,int)>();
        if(courant.Item1==r || courant.Item2==r) candidates.Add(courant);
        else { candidates.Add((Math.Min(r,courant.Item2),Math.Max(r,courant.Item2))); candidates.Add((Math.Min(r,courant.Item1),Math.Max(r,courant.Item1))); }
        var choix=candidates.OrderBy(a=>DistanceConfig(courant,a,d)+suivant[a]).First();
        cout+=DistanceConfig(courant,choix,d); courant=choix; w=suivant;
    }
    return (cout,w.Values.Min());
}
double JouerGlouton((int,int) init,int[] req,Func<int,int,int> d) {
    int[] p={init.Item1,init.Item2}; double cout=0;
    foreach(int r in req) { int i=d(p[0],r)<=d(p[1],r)?0:1; cout+=d(p[i],r); p[i]=r; }
    return cout;
}
Func<int,int,int> cycle=(i,j)=>Math.Min(Math.Abs(i-j),5-Math.Abs(i-j));
Func<int,int,int> ligne=(i,j)=>Math.Abs(i-j);
foreach(int repetitions in new[]{30,60,240}) {
    var req=Enumerable.Range(0,2*repetitions).Select(i=>i%2==0?3:4).ToArray();
    var x=JouerWfa(5,(0,1),req,cycle); double g=JouerGlouton((0,1),req,cycle);
    Console.WriteLine($"C5 T={req.Length} : OPT={x.opt}, WFA={x.wfa} ratio={x.wfa/x.opt:F3}, glouton={g} ratio={g/x.opt:F1}");
}
var alea=Enumerable.Range(0,60).Select(_=>rng.Next(8)).ToArray(); var l8=JouerWfa(8,(0,7),alea,ligne);
Console.WriteLine($"Ligne 8, T=60 : OPT={l8.opt}, WFA/OPT={l8.wfa/l8.opt:F3}, glouton/OPT={JouerGlouton((0,7),alea,ligne)/l8.opt:F3}");

C5 T=60 : OPT=3, WFA=5 ratio=1.667, glouton=61 ratio=20.3


C5 T=120 : OPT=3, WFA=5 ratio=1.667, glouton=121 ratio=40.3


C5 T=480 : OPT=3, WFA=5 ratio=1.667, glouton=481 ratio=160.3


Ligne 8, T=60 : OPT=79, WFA/OPT=1.101, glouton/OPT=1.101


Sur le cycle, la séquence alternée est un adversaire du plus proche. Les ratios finis sont mesurés sur ce banc ; une garantie compétitive générale inclut éventuellement une constante additive et ne découle pas de ces seuls essais.

## Exercice 2 — Étoile et stratégie gloutonne

Construire la distance de l'étoile à m feuilles puis chercher une séquence qui coûte cher au plus proche. Appeler WFA avec m+1 points ; comparer les coûts et l'OPT.

In [7]:
int? DistanceEtoile(int a,int b,int m) {
    // TODO : diagonale nulle, centre-feuille 1, deux feuilles distinctes 2.
    return null;
}
var ex2=DistanceEtoile(0,2,3);
if(ex2==null) Console.WriteLine("Exercice 2 à compléter");
else {
    if(ex2!=1 || DistanceEtoile(1,2,3)!=2 || DistanceEtoile(2,2,3)!=0) throw new Exception("Distance de l'étoile incorrecte");
    Console.WriteLine("Étoile : centre, feuilles distinctes et diagonale vérifiés");
}

Exercice 2 à compléter


## 4. Récapitulatif

La source Python confronte l'ordre aléatoire des secrétaires aux requêtes adversariales du k-server et rapporte les deux prépublications sans reproduire leurs preuves. Le récapitulatif rejoue ses essais supplémentaires.

In [8]:
Console.WriteLine($"Secrétaire n=1000, 5000 essais : {MonteCarlo(1000,5000,rng):F4}");
Console.WriteLine($"Uniforme : pur {Competitivite(valU,uniforme,false,2000,rng):F4}, seuil {Competitivite(valU,uniforme,true,2000,rng):F4}");
Console.WriteLine("C5 : voir les trois longueurs et les coûts réels ci-dessus");

Secrétaire n=1000, 5000 essais : 0.3738


Uniforme : pur 0.6982, seuil 0.7283


C5 : voir les trois longueurs et les coûts réels ci-dessus


## Exercice 3 — Valeurs log-normales

Tirer n valeurs log-normales indépendantes, appliquer la règle du secrétaire et estimer le succès. Expliquer pourquoi l'ordre relatif reste uniforme pour une loi continue.

In [9]:
bool? SecretaireLogNormale(int n,double mu,double sigma,Random r) {
    // TODO : générer des normales (par exemple Box–Muller), exponentier, appliquer la règle.
    return null;
}
var ex3=SecretaireLogNormale(100,0,1,new Random(4));
if(ex3==null) Console.WriteLine("Exercice 3 à compléter");
else {
    var rEx=new Random(99); int victoires=0;
    for(int i=0;i<3000;i++) if(SecretaireLogNormale(100,0,1,rEx)==true) victoires++;
    double taux=victoires/3000.0;
    if(taux<0.32 || taux>0.42) throw new Exception("Fréquence de succès incohérente avec le seuil de rang");
    Console.WriteLine($"Secrétaire log-normal : succès={taux:F3} sur 3000 essais");
}

Exercice 3 à compléter


## À retenir

- Les règles en ligne décident sans connaître la suite ; le secrétaire suppose une arrivée aléatoire, k-server admet une suite adversariale.
- Les deux oracles rendent explicites les contraintes ; WFA et OPT partagent une DP exacte sur les petites configurations.
- Une mesure finie et une garantie générale sont deux objets distincts.

## Vérifiez votre compréhension

1. Pourquoi le seuil du secrétaire ne dépend-il que des rangs ?
2. Pourquoi les deux matroïdes peuvent-ils favoriser des stratégies différentes ?
3. Qu'apporte la fonction de travail au choix en ligne ?

<details><summary>Éléments de réponse</summary>

1. Seules les comparaisons d'ordre interviennent. 2. Les contraintes changent les ensembles admissibles. 3. Elle résume les coûts optimaux des préfixes sans utiliser les requêtes futures.
</details>

## Pour aller plus loin

[Notebook Python source](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) ; [secrétaire matroïdal](Complexity-04d-Secretaire-Matroidal-Python.ipynb). Les prépublications sont citées par le source Python ; leurs preuves n'ont pas été consultées pour ce port.